# Práctica 2: Reasoning y Function Calling en Azure AI Foundry
**Estudiante:** Borja Núñez Salegui  
**Curso:** Máster en IA & Big Data Engineering - Tajamar Tech

Este notebook explora las capacidades de razonamiento profundo de los modelos de última generación y la integración de herramientas externas mediante Function Calling.

## 0. Configuración del Entorno
Cargamos las credenciales necesarias para conectar con el endpoint de Azure AI Foundry.

In [1]:
import os
import json
from openai import AzureOpenAI
from dotenv import load_dotenv

load_dotenv()

client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_KEY"),
    api_version="2024-08-01-preview"
)

deployment_name = "gpt-4o"

## 2.1 Razonamiento (Reasoning)
Los modelos de razonamiento (como `gpt-4.5-preview`, `o1` u `o3-mini`) permiten configurar el esfuerzo de razonamiento (`reasoning_effort`). 

### Comparativa de niveles de razonamiento
A continuación, ejecutamos el modelo utilizando el parámetro nativo `reasoning_effort` para evaluar su profundidad.

In [ ]:
prompt_mecanica = "¿Por qué un motor 1.4 TDCi de 2005 puede perder potencia repentinamente?"

deployment_reasoning = "gpt-5.4-mini"

niveles_esfuerzo = ["low", "high"]

for esfuerzo in niveles_esfuerzo:
    print(f"\n--- ESFUERZO DE RAZONAMIENTO: {esfuerzo.upper()} ---")
    try:
        response = client.chat.completions.create(
            model=deployment_reasoning,
            messages=[
                {"role": "user", "content": prompt_mecanica}
            ],
            reasoning_effort=esfuerzo
        )
        print(response.choices[0].message.content)
    except Exception as e:
        print(f"Error al procesar: {e}")

## 2.2 Function Calling
Conectamos la IA con una función de base de datos técnica.

In [3]:
def check_engine_specs(motor):
    specs = {"1.4 tdci": "68 CV, 160 Nm.", "1.6 tdci": "90/110 CV."}
    return specs.get(motor.lower(), "No encontrado.")

tools = [{
    "type": "function",
    "function": {
        "name": "check_engine_specs",
        "description": "Obtiene especificaciones técnicas de motores.",
        "parameters": {"type": "object", "properties": {"motor": {"type": "string"}}, "required": ["motor"]}
    }
}]

response = client.chat.completions.create(model=deployment_name, messages=[{"role": "user", "content": "Par motor del 1.4 TDCi"}], tools=tools)

if response.choices[0].message.tool_calls:
    for tool_call in response.choices[0].message.tool_calls:
        import json
        args = json.loads(tool_call.function.arguments)
        resultado = check_engine_specs(args['motor'])
        print(f"Resultado función: {resultado}")

Resultado función: 68 CV, 160 Nm.


## Conclusiones
1. **Latencia:** El razonamiento profundo requiere más tiempo.
2. **Precisión:** Las funciones garantizan datos reales.